In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 准备文档

In [2]:
# 1.读取文档
with open('./resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

In [3]:
# 2.切分文档
headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 是否去除文本中的标题，False：不去除
)
# list[Document]
documents = markdown_splitter.split_text(doc)
print(len(documents))

21


In [4]:
# 3.给文档添加ID
for i, document in enumerate(documents, start=1):
    document.id = f"doc_{i}"

# 初始化向量数据库

In [5]:
# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 向量维度，可选：减少维度以节省存储
)

In [6]:
# 2.初始化VectorStore
vector_store = InMemoryVectorStore(ollama_embeddings)

# 添加和删除文档

In [7]:
# 批量添加文档，返回文档id
ids = vector_store.add_documents(documents)

In [8]:
# 根据id删除文档
vector_store.delete(["doc_1", "doc_2"])

# 向量检索文档

In [11]:
# 1.普通搜索，结果是一个list[Document]
result = vector_store.similarity_search(
    query="论语中教育的目的是什么",
    k=3
)
for document in result:
    print(f"=============={document.id}===================")
    print(document.model_dump_json(indent=2))

==============doc_3===================
{
  "id": "doc_3",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
==============doc_4===================
{
  "id": "doc_4",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}
==============doc_5===================
{
  "id": "doc_5",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。",
  "type": "Document"
}


In [12]:
# 2.带分数查询，结果是list[tuple[Document, float]]
result = vector_store.similarity_search_with_score(
    query="论语中教育的目的是什么",
    k=3
)

for document, score in result:
    print(f"=============={document.id}: {score}===================")
    print(document.model_dump_json(indent=2))

==============doc_3: 0.7214450312388188===================
{
  "id": "doc_3",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
==============doc_4: 0.6494818818743939===================
{
  "id": "doc_4",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}
==============doc_5: 0.6222374570348406===================
{
  "id": "doc_5",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。